In [ ]:
import os
import sys
notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(notebook_dir))

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

import logging
import json
from matplotlib import gridspec
from matplotlib.patches import Patch
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from force_regression.config.dataconfig import DataConfig
import force_regression.utils.functions as fn
import force_regression.evaluation.compile_results as cr
from typing import List
from configs.constants import *
print(f'Notebook dir: {notebook_dir}\nProject dir: {project_dir}.')

logging.getLogger().setLevel(logging.INFO)
%load_ext autoreload
%autoreload 2

In [ ]:
config_path = os.path.join(project_dir, 'configs/config.json')

try:
    with open(config_path, 'r') as config_file:
        config = json.load(config_file)
        root_dir = os.path.join(project_dir, config['root_dir'])
        root_results_dir = os.path.join(project_dir, config['root_results_dir'])
        subject_mappings = config['subject_mappings']
        print(f'Root directory from config: {root_dir}')
except FileNotFoundError:
    print(f"Error: 'config.json' not found in {config_path}")
except json.JSONDecodeError:
    print("Error: 'config.json' is not a valid JSON file.")
except KeyError:
    print("Error: 'root_dir' not found in 'config.json'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

results_dir = 'output_files'

In [ ]:
# used to retrieve the figs_dir
data_config = DataConfig(root_dir=root_dir,
                         root_results_dir = root_results_dir,
                          subject=fn.reverse_remap("S1", subject_mappings), 
                          task_type="Trap", finger_type="Individual", day="Day 1",
                            emg_type="surf",
                            f_samp=10240,
                            subj_map=subject_mappings,
                            segment_hold=True, 
                            verbose=False, 
                            common_only=True, 
                            images=False,
                            time_to_cut=1,
                            results_dir=results_dir)
output_figures_dir = os.path.join(data_config.root_results_dir, data_config.figs_dir)


# Load data

In [ ]:
subjects = ["S1", "S2"]
# load baseline data
metrics_df = cr.load_and_parse_df_data(subject_list=subjects,
                                       data_type='metrics_df',
                                       config=config,
                                       results_dir=results_dir,
                                       is_sweep=True)


In [ ]:
# for each window size and input type, sign_mvc, there should be 10 entries: 2 folds x 5 fingers (fingers activated sequentially)
metrics_df.groupby([SUBJECT_COL,INPUT_TYPE_COL, WS_COL,SIGN_MVC_COL ])[WS_COL].count()


# Aggregate results

In [ ]:
plot_metric = 'RMSE_test_post'
select_mvc = 15

# Compute mean and std the metrics for each window and subjec
mean_std_df_per_dir = metrics_df.groupby([SUBJECT_COL,INPUT_TYPE_COL, WS_COL,SIGN_MVC_COL ])[plot_metric].agg(['mean', 'std']).reset_index()
mean_std_df = metrics_df.groupby([SUBJECT_COL,INPUT_TYPE_COL, WS_COL ])[plot_metric].agg(['mean', 'std']).reset_index()
mean_std_df_per_dir

# Plot sweep results per direction

In [ ]:
def plot_sweep_per_subject_with_directions(mean_std_df_per_dir: pd.DataFrame,
                                            subjects: List[str],
                                            plot_metric: str = 'RMSE_test',
                                            select_input_type: str = 'MU spike count',
                                            save_fig: bool = False,
                                            output_figures_dir: str = None,
                                            fig_name: str = None):
    """
    Line plot of average RMSE vs window size per subject, with flexion and extension
    overlaid in each subplot and fill_between for std.
    """
    direction_colors = {1: COLORS['burgundy'], -1: COLORS['belize']}
    direction_labels = {1: 'Flexion', -1: 'Extension'}
    
    mean_col = 'mean'
    std_col = 'std'
    
    fig = plt.figure(figsize=(10, 3))
    gs = gridspec.GridSpec(nrows=1, ncols=len(subjects))
    
    for col, subject in enumerate(subjects):
        ax = fig.add_subplot(gs[col])
        subject_data = mean_std_df_per_dir[
            (mean_std_df_per_dir[SUBJECT_COL] == subject) &
            (mean_std_df_per_dir[INPUT_TYPE_COL] == select_input_type)
        ]
        
        for sign_mvc in sorted(subject_data[SIGN_MVC_COL].unique()):
            y_data = subject_data[subject_data[SIGN_MVC_COL] == sign_mvc].copy()
            y_data[WS_COL] = pd.to_numeric(y_data[WS_COL], errors='coerce') * 1000
            y_data = y_data.sort_values(WS_COL)
            
            color = direction_colors[sign_mvc]
            label = direction_labels[sign_mvc]
            
            ax.plot(y_data[WS_COL], y_data[mean_col],
                    marker='o', ms=5, color=color, label=label)
            ax.fill_between(y_data[WS_COL],
                            y_data[mean_col] - y_data[std_col],
                            y_data[mean_col] + y_data[std_col],
                            color=color, alpha=0.2)
        
        ax.set_title(f"Subject {subject.split('S')[-1]}", fontsize=TITLE_FONTSIZE)
        ax.set_xlabel('Window size (ms)', fontsize=LABEL_FONTSIZE)
        ax.set_ylabel(RMSE_LABEL, fontsize=LABEL_FONTSIZE, labelpad=YLAB_PAD)
        ax.set_xticks(sorted(subject_data[WS_COL].unique() * 1000))
        ax.set_ylim([0, 8])
        ax.tick_params(axis='both', labelsize=LABEL_FONTSIZE)
        sns.despine(ax=ax)
    
    # Shared legend
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles=handles, labels=labels,
               loc='upper center', bbox_to_anchor=(0.5, 1.1),
               ncol=2, frameon=False, fontsize=LABEL_FONTSIZE)
    
    fig.tight_layout()
    
    if save_fig and output_figures_dir:
        if fig_name is None:
            fig_name = f"sweep_ws_{plot_metric}_per_direction.png"
        fig_path = os.path.join(output_figures_dir, fig_name)
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved at {fig_path}")
    
    return None

In [ ]:
plot_sweep_per_subject_with_directions(
    mean_std_df_per_dir,
    subjects=['S1', 'S2'],
    plot_metric=plot_metric,
    select_input_type='MU spike count'
)

# Plot sweep results pooled across directions

In [ ]:
def plot_sweep_per_subject_across_directions(mean_std_df: pd.DataFrame,
                                              subjects: List[str],
                                              plot_metric: str = 'RMSE_test',
                                              save_fig: bool = False,
                                              output_figures_dir: str = None,
                                              fig_name: str = None):
    """
    Line plot of average RMSE vs window size per subject, pooled across directions.
    One line per input type with fill_between for std.
    """
    models_palette = {
        'global features': COLORS['burgundy'],
        'MU spike count': '#FF6F5C',
    }
    
    
    fig = plt.figure(figsize=(8, 2.5))
    gs = gridspec.GridSpec(nrows=1, ncols=len(subjects))
    chosen_ws = 0.08  # 0.08s

    for col, subject in enumerate(subjects):
        ax = fig.add_subplot(gs[col])
        subject_data = mean_std_df[mean_std_df[SUBJECT_COL] == subject]
        
        for input_type in subject_data[INPUT_TYPE_COL].unique():
            y_data = subject_data[subject_data[INPUT_TYPE_COL] == input_type].copy()
            y_data[WS_COL] = pd.to_numeric(y_data[WS_COL], errors='coerce') 
            y_data = y_data.sort_values(WS_COL)
            
            color = models_palette.get(input_type, COLORS['midnight_blue'])
            
            ax.plot(y_data[WS_COL], y_data['mean'],
                    marker='o', ms=5, color=color, label=input_type)
            ax.fill_between(y_data[WS_COL],
                            y_data['mean'] - y_data['std'],
                            y_data['mean'] + y_data['std'],
                            color=color, alpha=0.2)
        
        ax.set_title(f"Subject {subject.split('S')[-1]}", fontsize=TITLE_FONTSIZE)
        ax.set_xlabel('Window size (s)', fontsize=LABEL_FONTSIZE)
        ax.set_ylabel(RMSE_LABEL, fontsize=LABEL_FONTSIZE, labelpad=YLAB_PAD)
        xticks = sorted(subject_data[WS_COL].unique())
        ax.set_xticks(xticks[::2])  # show every other tick
        
        # Get RMSE at chosen window size for each input type and draw horizontal line + annotation
        for input_type in subject_data[INPUT_TYPE_COL].unique():
            y_data = subject_data[subject_data[INPUT_TYPE_COL] == input_type].copy()
            y_data[WS_COL] = pd.to_numeric(y_data[WS_COL], errors='coerce')
            rmse_at_ws = y_data.loc[y_data[WS_COL] == chosen_ws, 'mean']
            std_at_ws = y_data.loc[y_data[WS_COL] == chosen_ws, 'std']
            if not rmse_at_ws.empty:
                rmse_val = rmse_at_ws.values[0]
                std_val = std_at_ws.values[0]
                color =  COLORS['midnight_blue']

                # Horizontal line from y-axis to the point
                ax.plot([0, chosen_ws], [rmse_val, rmse_val],
                        color=color, linestyle=':', linewidth=1, alpha=0.6)
                # # Vertical line from x-axis to the point
                ax.plot([chosen_ws, chosen_ws], [0, rmse_val],
                        color=color, linestyle=':', linewidth=1, alpha=0.6)
                ax.plot(chosen_ws, rmse_val, marker='o', ms=5, color=color)  # mark the point

        # ax.set_xticks(sorted(subject_data[WS_COL].unique()))
        ax.set_ylim([0, 5.2])
        ax.set_xlim([0.005,0.22])
        ax.tick_params(axis='both', labelsize=LABEL_FONTSIZE-2)
        sns.despine(ax=ax)
    fig.tight_layout()
    
    if save_fig and output_figures_dir:
        if fig_name is None:
            fig_name = f"sweep_ws_{plot_metric}_across_directions.png"
        fig_path = os.path.join(output_figures_dir, fig_name)
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved at {fig_path}")
    
    return None


In [ ]:
plot_sweep_per_subject_across_directions(
    mean_std_df,
    subjects=['S1', 'S2'],
    plot_metric='RMSE_test',
    save_fig=False,
    output_figures_dir=output_figures_dir,
)
